# 01 · Data sources: from a public API to `Evidence`

**What you will learn**
1. The `Evidence` model: the one shape every source returns
2. The shared `HttpClient` (retries + cache) and the ACID `Store` behind the cache
3. Each of the four sources, called directly
4. Why the cache matters (speed, fewer calls, same answer)

```
openFDA labels ─┐
openFDA FAERS ──┤   source.search(...)  ──►  list[Evidence]  ──►  tools (notebook 02)  ──►  agents (notebook 03)
ClinicalTrials ─┤
PubMed ─────────┘
```

In [ ]:
# Setup: make the project importable and quiet the HTTP logs
import sys, logging, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.getLogger("httpx").setLevel(logging.WARNING)

def show(obj, limit=1500):
    """Pretty-print a pydantic model, dict or list, cut to `limit` characters."""
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    if isinstance(obj, list):
        obj = [o.model_dump() if hasattr(o, "model_dump") else o for o in obj]
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n... (cut)")

## 1. Config
Everything is built from `configs/default.yaml`. The config is a plain dict, so you can tweak it here.
Using `store.path = ":memory:"` keeps this notebook's data out of the real run log.

In [ ]:
from pharma_agent.config import load_config
cfg = load_config(overrides={"store.path": ":memory:"})
print("Sources to load:", cfg["sources"])
print("LLM roles:", cfg["llm"]["roles"])

## 2. `Evidence`: the contract between sources and agents
Every source returns a list of these. The `id` is what the final answer cites, and the guardrail later checks
that every cited id really came from a tool.

In [ ]:
from pharma_agent.schemas import Evidence
print(list(Evidence.model_json_schema()["properties"]))

## 3. The HTTP client and the ACID store
`build_http` gives one client shared by all sources (dependency injection). It retries on 429/5xx with
exponential back-off and caches answers in SQLite. Each cache write is a transaction, so a crash never
leaves a half-written row.

In [ ]:
from pharma_agent.wiring import build_http, build_store
from pharma_agent.sources import build_sources

store = build_store(cfg)            # SQLite (in memory here)
http = build_http(cfg, store)       # retries + cache
sources = {s.name: s for s in build_sources(cfg["sources"], http)}
for name, s in sources.items():
    print(f"{name:26s} <- {type(s).__name__}")

## 4. Each source, called directly
### FDA drug label: one section (here, the boxed warning and warnings)

In [ ]:
[label] = sources["search_drug_labels"].search(drug="metformin", section="warnings")
print(label.id)
print(label.title)
print(label.url)
print(label.metadata)
print(label.text[:500])

### FAERS: most reported adverse reactions (reports, not proof of causation)

In [ ]:
[faers] = sources["get_adverse_event_counts"].search(drug="atorvastatin", top_n=8)
print(faers.text)

### ClinicalTrials.gov: filter by drug, condition and status

In [ ]:
trials = sources["search_clinical_trials"].search(condition="obesity", intervention="semaglutide", status="RECRUITING", limit=3)
for t in trials:
    print(t.id, "|", t.title[:90])
    print("   ", t.text[:200], "\n")

### PubMed: search, then fetch abstracts

In [ ]:
papers = sources["search_pubmed"].search(query="SGLT2 inhibitors heart failure outcomes", limit=3)
for p in papers:
    print(p.id, "|", p.title[:100])

### Nothing found is a normal answer, not an error

In [ ]:
print(sources["search_drug_labels"].search(drug="notarealdrugxyz"))

## 5. The cache in action
The second identical request is served from SQLite: same answer, no network call.

In [ ]:
import time
url, params = "https://api.fda.gov/drug/label.json", {"search": 'openfda.generic_name:"warfarin"', "limit": 1}
for attempt in (1, 2):
    t0 = time.perf_counter()
    res = http.get(url, params)
    print(f"call {attempt}: status={res.status} from_cache={res.from_cache} {1000 * (time.perf_counter() - t0):.0f} ms")
print("rows in http_cache:", store.count("http_cache"))

## Try it yourself
* Change `section=` to `"drug_interactions"`, `"pregnancy"` or `"mechanism"`.
* Search trials with `status="COMPLETED"`.
* Next: **02_tools_and_mcp.ipynb** turns these sources into tools.